# DataLens Exploration

This notebook walks through the `datalens` package end to end on the bundled
sample dataset (`data/sample.csv`): a synthetic coffee shop's sales log with
columns `date`, `store`, `category`, `item`, `quantity`, `unit_price`, and
`revenue`.

The dataset is deliberately a little messy - it has a handful of duplicate
rows and missing values baked in (see `scripts/generate_sample_data.py`) -
so the cleaning functions below have real work to do, the same as they
would on a real export.

We'll:
1. Load and preview the raw data
2. Clean it with `datalens.cleaning`
3. Summarise it with `datalens.analysis`
4. Chart revenue by category and over time


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the sibling `datalens` package importable when running this notebook
# straight out of the repo (no install required).
sys.path.insert(0, str(Path.cwd().parent))

from datalens.analysis import group_by_summary, rolling_average, summarize
from datalens.cleaning import clean_data

%matplotlib inline


## 1. Load and preview the raw data

In [ ]:
root = Path.cwd()
file = "sample.csv"
while not (root / "data" / file).exists() and root != root.parent:
    root = root.parent

filepath = root / "data" / file
if not filepath.exists():
    raise FileNotFoundError(f"Could not find {file}")

raw = pd.read_csv(filepath)
print(f"{len(raw)} rows, {len(raw.columns)} columns")
raw.head()

## 2. Clean the data

`clean_data` runs the standard pipeline: coerce column types (dates and
numerics), drop exact-duplicate rows, then handle missing values (default
strategy is to drop rows with any missing field - see
`datalens.cleaning.handle_missing_values` for a `"fill"` strategy that
imputes instead).

In [ ]:
cleaned = clean_data(raw)
print(f"raw: {len(raw)} rows -> cleaned: {len(cleaned)} rows "
      f"({len(raw) - len(cleaned)} rows removed as duplicates or incomplete)")
cleaned.head()


## 3. Summary statistics

In [ ]:
summary = summarize(cleaned)
for key, value in summary.items():
    print(f"{key}: {value}")


## 4. Revenue by category

In [ ]:
by_category = group_by_summary(cleaned, by="category", value_column="revenue")
by_category


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
by_category["total_revenue"].plot(kind="bar", ax=ax, color="#1E62C2")
ax.set_title("Total revenue by category",fontsize=13,fontweight='bold')
ax.set_xlabel("Category")
ax.set_ylabel("Total Revenue ($)")
fig.tight_layout()
plt.show()


## 5. Revenue trend over time

`rolling_average` groups revenue by day and applies a rolling mean, which
smooths out day-to-day noise so the underlying trend is easier to read.

In [ ]:
trend = rolling_average(cleaned, column="revenue", window=7, date_column="date")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend.index, trend["revenue"], alpha=0.35,color = "#1740a0", label="Daily Revenue")
ax.plot(trend.index, trend["revenue_rolling_avg"], color="#C72429", linewidth=2,
        label="7-Day Rolling Average")
ax.set_title("Daily Revenue with 7-Day Rolling Average",fontsize="13",fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue ($)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## Next steps

This only scratches the surface of what's in `datalens`. Some ideas worth
exploring, several of which are open as issues in this repo:

- Outlier detection on `revenue` or `quantity` per category
- A `datalens report` CLI command that writes this kind of summary to a
  markdown file automatically
- A second sample dataset (different schema) to test reconciliation logic
  against
- More chart types - a line chart over time is a natural first PR

See `CONTRIBUTING.md` for how to claim an issue and get a PR up quickly.